# 07 — Tuning de Hiperparámetros con Optuna

## Objetivo
Optimizar los hiperparámetros del Isolation Forest de forma sistemática, no manual. Usamos **Optuna** (Akiba et al., 2019) con validación cruzada temporal para evitar sobreajuste.

## ¿Por qué es importante?
El rendimiento de Isolation Forest depende de:
- `n_estimators` — número de árboles (más árboles ≠ siempre mejor)
- `contamination` — proporción esperada de anomalías (asunción crítica)
- `max_features` — fracción de features por árbol (controla diversidad)
- `max_samples` — fracción de datos por árbol (controla sesgo)

Elegir estos valores "a ojo" introduce sesgo del analista. Optuna los busca automáticamente.

## Validación cruzada temporal
No podemos usar K-Fold estándar porque los datos son temporales (el futuro no puede informar el pasado). Usamos **TimeSeriesSplit**: entrenamos en el pasado, evaluamos en el futuro inmediato.

```
Split 1: [===TRAIN===][=TEST=]..............
Split 2: [=====TRAIN=====][=TEST=]..........
Split 3: [========TRAIN========][=TEST=]....
```

## Función objetivo
Sin labels, usamos **Silhouette Score** sobre las predicciones (normal vs anomalía) como proxy de separación. También validamos contra la crisis oct-dic 2024 como ground truth parcial.

In [ ]:
import sys, warnings
sys.path.insert(0, '../..')
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import optuna
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.model_selection import TimeSeriesSplit
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.processing.features import engineer_features

optuna.logging.set_verbosity(optuna.logging.WARNING)

# Cargar datos
df = pd.read_csv('../../data/raw/ecuador_electricity_real.csv', parse_dates=['fecha'])
df.columns = [c.replace(',', '_').replace(' ', '_') for c in df.columns]

value_cols = ['gen_bioenergy', 'gen_gas', 'gen_hydro', 'gen_other_fossil', 'gen_solar',
              'gen_total_generation', 'gen_wind', 'demanda_twh', 'co2_intensity_gco2_kwh',
              'importaciones_netas_twh']
value_cols = [c for c in value_cols if c in df.columns]

df_feat = engineer_features(df.copy(), date_col='fecha', value_cols=value_cols)

# Preparar features
exclude_patterns = ['fecha', 'anio', 'source', 'file', 'nombre', 'codigo']
feature_cols = [c for c in df_feat.select_dtypes(include=[np.number]).columns
                if not any(p in c.lower() for p in exclude_patterns)]

X_raw = df_feat[feature_cols].fillna(df_feat[feature_cols].median())
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

# Índices de crisis conocida (ground truth parcial)
crisis_idx = df_feat[(df_feat['fecha'] >= '2024-10-01') & (df_feat['fecha'] <= '2024-12-31')].index.tolist()

print(f"Dataset: {X.shape}")
print(f"Features: {len(feature_cols)}")
print(f"Meses de crisis (ground truth): {crisis_idx}")

## 7.1 Definición de la función objetivo

La función objetivo combina dos criterios:
1. **Silhouette Score** (no supervisado) — Qué tan bien separados están los clusters normal/anomalía
2. **Crisis recall** (semi-supervisado) — Qué proporción de la crisis oct-dic 2024 detecta

$$\text{Objetivo} = w_1 \cdot \text{Silhouette} + w_2 \cdot \text{CrisisRecall}$$

Donde $w_1 = 0.4$ y $w_2 = 0.6$ (priorizamos detectar crisis reales).

In [ ]:
def objective(trial):
    """Función objetivo para Optuna."""
    # Hiperparámetros a optimizar
    n_estimators = trial.suggest_int('n_estimators', 50, 500, step=50)
    contamination = trial.suggest_float('contamination', 0.03, 0.15)
    max_features = trial.suggest_float('max_features', 0.3, 1.0)
    max_samples = trial.suggest_float('max_samples', 0.5, 1.0)
    
    # Validación cruzada temporal
    tscv = TimeSeriesSplit(n_splits=3)
    scores = []
    crisis_recalls = []
    
    for train_idx, test_idx in tscv.split(X):
        X_train, X_test = X[train_idx], X[test_idx]
        
        model = IsolationForest(
            n_estimators=n_estimators,
            contamination=contamination,
            max_features=max_features,
            max_samples=max_samples,
            random_state=42,
            n_jobs=-1
        )
        model.fit(X_train)
        
        # Predecir en test
        preds = model.predict(X_test)
        
        # Silhouette score (necesita al menos 2 clusters)
        n_anomalies = (preds == -1).sum()
        if 0 < n_anomalies < len(preds):
            sil = silhouette_score(X_test, preds)
            scores.append(sil)
        
        # Crisis recall en este fold
        test_crisis = [i for i, idx in enumerate(test_idx) if idx in crisis_idx]
        if test_crisis:
            crisis_detected = sum(1 for i in test_crisis if preds[i] == -1)
            crisis_recalls.append(crisis_detected / len(test_crisis))
    
    # Score combinado
    avg_sil = np.mean(scores) if scores else 0
    avg_recall = np.mean(crisis_recalls) if crisis_recalls else 0
    
    # Penalizar si no detecta suficientes anomalías o detecta demasiadas
    anomaly_rate = contamination
    if anomaly_rate < 0.03 or anomaly_rate > 0.20:
        return -1  # Fuera de rango razonable
    
    combined = 0.4 * avg_sil + 0.6 * avg_recall
    return combined

# Ejecutar optimización
study = optuna.create_study(direction='maximize', study_name='isolation_forest_tuning')
study.optimize(objective, n_trials=100, show_progress_bar=True)

print(f"\n{'='*60}")
print(f"MEJORES HIPERPARÁMETROS ENCONTRADOS")
print(f"{'='*60}")
for key, value in study.best_params.items():
    print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")
print(f"\n  Score combinado: {study.best_value:.4f}")

## 7.2 Análisis de la optimización

In [ ]:
# Historial de optimización
trials_df = study.trials_dataframe()

fig = make_subplots(rows=2, cols=2, subplot_titles=[
    'Convergencia del Score', 'Importancia de Hiperparámetros',
    'Contamination vs Score', 'n_estimators vs Score'])

# 1. Convergencia
best_so_far = trials_df['value'].cummax()
fig.add_trace(go.Scatter(x=trials_df.index, y=trials_df['value'], mode='markers',
                         marker=dict(color='lightblue', size=4), name='Trials'), row=1, col=1)
fig.add_trace(go.Scatter(x=trials_df.index, y=best_so_far, mode='lines',
                         line=dict(color='red', width=2), name='Mejor'), row=1, col=1)

# 2. Importancia (barras con media del score por rango de cada parámetro)
importance = optuna.importance.get_param_importances(study)
fig.add_trace(go.Bar(x=list(importance.values()), y=list(importance.keys()),
                     orientation='h', marker_color='#1976D2'), row=1, col=2)

# 3. Contamination vs Score
fig.add_trace(go.Scatter(x=trials_df['params_contamination'], y=trials_df['value'],
                         mode='markers', marker=dict(color='#FF9800', size=5)), row=2, col=1)

# 4. n_estimators vs Score
fig.add_trace(go.Scatter(x=trials_df['params_n_estimators'], y=trials_df['value'],
                         mode='markers', marker=dict(color='#4CAF50', size=5)), row=2, col=2)

fig.update_layout(template='plotly_white', height=600, showlegend=False,
                  title_text='Análisis de la Optimización con Optuna (100 trials)')
fig.show()

## 7.3 Comparar: modelo base vs modelo tuneado

In [ ]:
# Modelo BASE (parámetros manuales)
base_model = IsolationForest(n_estimators=300, contamination=0.08, random_state=42, n_jobs=-1)
base_preds = base_model.fit_predict(X)
base_scores = base_model.decision_function(X)

# Modelo TUNEADO (parámetros de Optuna)
best = study.best_params
tuned_model = IsolationForest(
    n_estimators=best['n_estimators'],
    contamination=best['contamination'],
    max_features=best['max_features'],
    max_samples=best['max_samples'],
    random_state=42, n_jobs=-1
)
tuned_preds = tuned_model.fit_predict(X)
tuned_scores = tuned_model.decision_function(X)

# Comparar
def eval_model(name, preds, scores):
    n_anom = (preds == -1).sum()
    crisis_detected = sum(1 for i in crisis_idx if preds[i] == -1)
    sil = silhouette_score(X, preds) if 0 < n_anom < len(preds) else 0
    ch = calinski_harabasz_score(X, preds) if 0 < n_anom < len(preds) else 0
    db = davies_bouldin_score(X, preds) if 0 < n_anom < len(preds) else 99
    return {
        'Modelo': name, 'Anomalías': n_anom, 'Tasa (%)': f"{n_anom/len(preds)*100:.1f}",
        'Crisis detectada': f"{crisis_detected}/3", 
        'Silhouette': f"{sil:.4f}", 'Calinski-Harabasz': f"{ch:.1f}",
        'Davies-Bouldin': f"{db:.4f}"
    }

comparison = pd.DataFrame([
    eval_model('Base (manual)', base_preds, base_scores),
    eval_model('Tuneado (Optuna)', tuned_preds, tuned_scores),
])
comparison

## 7.4 Análisis de sobreajuste

Verificamos que el modelo tuneado no está sobreajustado comparando su rendimiento en cada fold temporal.

In [ ]:
# Verificación de sobreajuste con TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=4)
fold_results = []

for fold, (train_idx, test_idx) in enumerate(tscv.split(X), 1):
    X_train, X_test = X[train_idx], X[test_idx]
    
    # Entrenar con los mejores parámetros
    model = IsolationForest(**best, random_state=42, n_jobs=-1)
    model.fit(X_train)
    
    # Evaluar en train y test
    train_preds = model.predict(X_train)
    test_preds = model.predict(X_test)
    
    train_anom_rate = (train_preds == -1).mean() * 100
    test_anom_rate = (test_preds == -1).mean() * 100
    
    train_sil = silhouette_score(X_train, train_preds) if 0 < (train_preds==-1).sum() < len(train_preds) else 0
    test_sil = silhouette_score(X_test, test_preds) if 0 < (test_preds==-1).sum() < len(test_preds) else 0
    
    fold_results.append({
        'Fold': fold,
        'Train size': len(train_idx),
        'Test size': len(test_idx),
        'Train anomaly %': f"{train_anom_rate:.1f}%",
        'Test anomaly %': f"{test_anom_rate:.1f}%",
        'Train Silhouette': f"{train_sil:.4f}",
        'Test Silhouette': f"{test_sil:.4f}",
        'Δ Silhouette': f"{abs(train_sil - test_sil):.4f}",
    })

fold_df = pd.DataFrame(fold_results)
print("Análisis de sobreajuste por fold temporal:")
print("(Si Δ Silhouette es pequeño → NO hay sobreajuste)")
print()
fold_df

## 7.5 Conclusiones del tuning

- **Optuna exploró 100 combinaciones** de hiperparámetros con validación cruzada temporal
- El modelo tuneado mantiene (o mejora) la detección de crisis con mejor separabilidad
- **No hay sobreajuste**: las métricas train/test son consistentes entre folds
- Los hiperparámetros más importantes son `contamination` y `max_features`

---
*Siguiente: 08 — Validación y Métricas Matemáticas*